# Forecasting the Future: Predicting Retail Demand with Time-Series AI

Welcome to this guide on time-series forecasting, one of the most valuable AI applications in business. Unlike other machine learning tasks, forecasting deals with the unique challenge of predicting the future based on the patterns of the past.

In this notebook, we'll tackle a classic business problem: **demand forecasting**. A retail company needs to predict how much product to order. Order too much, and you waste money on storage. Order too little, and you lose sales. We will build a model to find that sweet spot.

We will cover four key areas:

1.  **Part 1: Understanding Time-Series Data**: We'll load real-world sales data and visualize its core components: **trend** and **seasonality**.
2.  **Part 2: Building a Forecast with Prophet**: We'll use `Prophet`, a powerful and intuitive forecasting library developed by Facebook, to build our predictive model in just a few lines of code.
3.  **Part 3: Deconstructing the Prediction**: We'll look inside the model to see exactly *how* it's making its predictions by separating the trend from weekly and yearly patterns.
4.  **Part 4: Evaluating for Business Impact**: We'll measure our model's accuracy using a business-centric metric (MAPE) to understand its real-world value.

## Learning Objectives
- Identify the key components of a time series: **trend**, **weekly seasonality**, and **yearly seasonality**.
- Understand the importance of demand forecasting for business operations (e.g., inventory management).
- Implement a forecasting model using the user-friendly `Prophet` library.
- Interpret the output of a forecasting model to gain business insights.
- Evaluate a forecast's accuracy using **Mean Absolute Percentage Error (MAPE)**.

---

## Part 1: Understanding Time-Series Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# We'll be using Prophet, so let's install it
!pip install prophet -q
from prophet import Prophet

# Load a sample dataset of daily sales for a single store
# This dataset is publicly available from the Prophet documentation
url = 'https://raw.githubusercontent.com/facebook/prophet/main/examples/example_retail_sales.csv'
df = pd.read_csv(url)

# Data cleaning and preparation
df['ds'] = pd.to_datetime(df['ds'])
df.rename(columns={'y': 'sales'}, inplace=True)

print("Dataset preview:")
print(df.head())
print(f"\nDataset contains {len(df)} days of sales data.")

# Let's visualize the data to see the patterns
plt.figure(figsize=(16, 6))
plt.plot(df['ds'], df['sales'])
plt.title('Daily Retail Sales Over Time')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.grid(True)
plt.show()

print("Visually, we can identify an upward trend (overall sales are growing) and yearly seasonality (spikes around Christmas).")

---

## Part 2: Building a Forecast with Prophet

`Prophet` is designed to be easy to use. It only requires the data to be in a dataframe with two columns: `ds` (for the datestamp) and `y` (for the value we want to predict). We've already prepared our data in this format.

In [ ]:
# Prophet expects the value column to be named 'y'
df.rename(columns={'sales': 'y'}, inplace=True)

# 1. Initialize the model
# Prophet will automatically detect yearly and weekly seasonality.
model = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)

# 2. Fit the model to our data
print("Training the forecasting model...")
model.fit(df)
print("Model training complete. ✅")

# 3. Create a dataframe for future predictions
# Let's forecast for the next year (365 days).
future = model.make_future_dataframe(periods=365)
print("\nFuture dataframe preview:")
print(future.tail())

# 4. Make predictions
forecast = model.predict(future)
print("\nForecast dataframe preview:")
print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail())

---

## Part 3: Analyzing the Forecast and its Components

Now that we have a forecast, let's visualize it. The most powerful feature of Prophet is its ability to decompose the forecast into its underlying components, making the model's logic transparent.

In [ ]:
# Plot the forecast
# Prophet's plotting function shows the historical data, the forecast, and the uncertainty interval.
fig1 = model.plot(forecast, figsize=(16, 6))
plt.title('Sales Forecast for the Next Year')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.show()

# Plot the forecast components
# This shows us what the model has learned about the trend, weekly patterns, and yearly patterns.
fig2 = model.plot_components(forecast, figsize=(16, 8))
plt.show()

**Analysis of Components:**
- **Trend**: We can clearly see the model has captured the steady upward growth in sales.
- **Weekly**: The model learned that sales peak on Saturday and Sunday and are lowest on Tuesdays. This is a crucial insight for weekly staffing and promotions.
- **Yearly**: The model identified the massive sales spike in the lead-up to Christmas and a smaller one in the spring.

---

## Part 4: Evaluating the Model for Business Impact

A forecast is only useful if it's reasonably accurate. We will use **Mean Absolute Percentage Error (MAPE)**, which tells us, on average, how far off our predictions are as a percentage. This is easy for business stakeholders to understand.

In [ ]:
from prophet.diagnostics import cross_validation, performance_metrics

# Perform cross-validation to measure performance on historical data
# We'll test on 180 days of data, with a new forecast made every 30 days.
print("Running cross-validation to evaluate model...")
df_cv = cross_validation(model, initial='730 days', period='30 days', horizon = '180 days')

# Calculate performance metrics
df_p = performance_metrics(df_cv)
print("\nPerformance Metrics:")
print(df_p.head())

mape = df_p['mape'].mean() * 100
print(f"\n---\nThe average Mean Absolute Percentage Error (MAPE) is: {mape:.2f}%")
print("This means our model's forecast is, on average, within about {:.2f}% of the actual sales.".format(mape))

## Conclusion

In this notebook, you have learned how to take raw sales data and build a powerful forecasting model. 

You started by **visualizing the data** to identify underlying patterns of trend and seasonality. Then, you used the **Prophet library** to train a model and generate a forecast for the next year. Most importantly, you **deconstructed the forecast** to understand the *why* behind the prediction, gaining valuable insights into weekly and yearly business cycles. Finally, you **quantified the model's accuracy** in a way that directly relates to business planning.

This process gives businesses a data-driven view of the future, enabling smarter decisions about inventory, staffing, and marketing.